In [ ]:
#| default_exp bundle

In [ ]:
#| export
from __future__ import annotations
import hashlib, os, shutil, sys, zipfile
from fastcore.all import Path, first

In [ ]:
#| export
def graft(app, name, python=None):
    """Put the real `name` package into a built bundle, over whatever the freezer left of it.

    Some packages an archive simply cannot hold. `apsw` has the extension module for its own
    `__init__`, so a flattened copy loses `apsw.ext`; `playwright` carries a Node binary it has to
    *execute*, and a file inside an archive cannot be executed. Naming them in `packages` instead
    only gets `ImportError: No module named ...`, because the lookup wants an `__init__.py` to find.
    """
    import importlib.util
    spec = importlib.util.find_spec(name)
    if spec is None or not spec.submodule_search_locations:
        raise SystemExit(f'cannot graft {name}: it is not installed here')
    src = Path(first(spec.submodule_search_locations))
    lib = _lib_dir(app)
    # The package carries compiled modules built for the interpreter running this, so a bundle on
    # another version would take a wrong-ABI `.so` and fail at import with nothing to point at it.
    want = 'python%d.%d' % (python or sys.version_info[:2])
    if lib.name != want: raise SystemExit(f'cannot graft {name}: bundle is {lib.name}, this is {want}')
    dest = lib/name
    for stray in lib.glob(f'{name}.*'): stray.unlink()     # the flattened extension module
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(src, dest, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
    return dest.relative_to(app)

def _lib_dir(app):
    "The bundle's `lib/pythonX.Y`, on either platform's layout."
    app = Path(app)
    for base in (app/'Contents'/'Resources'/'lib', app/'lib', app):
        if (hit := first(sorted(base.glob('python3.*')))) is not None: return Path(hit)
    raise SystemExit(f'{app} has no lib/python3.* to write into')

In [ ]:
#| export
def link_duplicates(app, names, floor=1_000_000):
    """Hardlink identical files across the grafted packages. Returns the megabytes saved.

    `patchright` is a fork of `playwright` and ships the same 121MB Node binary, byte for byte, so
    grafting both writes it twice. A hardlink leaves two real, executable files and one copy of the
    bytes; these are read-only library files, so nothing can write through one and surprise the
    other.
    """
    lib = _lib_dir(app)
    seen, saved = {}, 0
    for name in names:
        for f in sorted((lib/name).rglob('*')):
            if not f.is_file() or f.is_symlink() or f.stat().st_size < floor: continue
            key = (f.stat().st_size, hashlib.sha256(f.read_bytes()).hexdigest())
            if (other := seen.get(key)) is None: seen[key] = f; continue
            if os.stat(f).st_ino == os.stat(other).st_ino: continue
            saved += f.stat().st_size
            f.unlink(); os.link(other, f)
    return round(saved / 1e6)

In [ ]:
#| export
def strip_zip(app, prefixes=()):
    """Rewrite the bundle's zip without entries nothing can read. Returns the megabytes saved.

    Anything grafted is in the archive twice over, and some entries were never reachable: a package
    that finds its own data with `Path(__file__).parent/...` and `.exists()` gets False for every
    path inside an archive, so those megabytes are carried and never opened.
    """
    z = first(sorted(_lib_dir(app).parent.glob('python3*.zip')))
    if not z or not prefixes: return 0
    prefixes = tuple(prefixes)
    keep, dropped = [], 0
    with zipfile.ZipFile(z) as src:
        for info in src.infolist():
            if info.filename.startswith(prefixes): dropped += info.compress_size; continue
            keep.append((info, src.read(info.filename)))
    if not dropped: return 0
    tmp = Path(str(z) + '.new')
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as out:
        for info, data in keep: out.writestr(info, data)
    tmp.replace(z)
    return round(dropped / 1e6)

In [ ]:
#| export
def install_modern_icon(app, icon, identity=None, deployment_target='14.0'):
    """Compile an Icon Composer document into a built bundle, after the freezer and before signing.

    The `iconfile` in the spec is the ICNS fallback, which every freezer installs and every macOS
    before 26 reads. The Icon Composer document is what 26 draws, and its plist entries must reach
    `Info.plist` before a signature covers them. None when the document or the tooling is absent:
    a machine without Xcode 26 still builds a working app, with the older icon.
    """
    icon = Path(icon) if icon else None
    if icon is None or not icon.is_dir(): return None
    try: from iconmage import install_icon
    except ImportError:
        print('  iconmage not installed; keeping the ICNS icon alone '
              '(uv tool install git+https://github.com/AnswerDotAI/iconmage.git)')
        return None
    try: return install_icon(icon, app, deployment_target=deployment_target, identity=identity)
    except Exception as e:
        print(f'  Icon Composer icon not installed ({type(e).__name__}: {e}); keeping the ICNS one')
        return None

In [ ]:
#| export
def finish(app, spec, identity=None):
    "Everything a bundle needs after the freezer wrote it, in the order it needs it."
    out = {}
    for name in spec.grafted: out[name] = str(graft(app, name))
    dropped = [*spec.unreachable, *[f'{n}/' for n in spec.grafted]]
    out['stripped_mb'] = strip_zip(app, dropped)
    out['linked_mb'] = link_duplicates(app, spec.grafted)
    out['modern_icon'] = bool(install_modern_icon(app, spec.modern_icon, identity))
    return out

In [ ]:
#| export
def frozen_distribution():
    """A setuptools `Distribution` with no `install_requires`, which py2app 0.28.10+ refuses.

    A build run from a project root makes setuptools read `pyproject.toml` and fill the field in
    from `[project] dependencies`. A frozen build wants none of it: the spec's `packages` and
    `includes` say what the bundle carries, because the scanner cannot see an import made inside a
    function.
    """
    from setuptools.dist import Distribution
    class Frozen(Distribution):
        def parse_config_files(self, *args, **kwargs):
            super().parse_config_files(*args, **kwargs)
            self.install_requires = []
    return Frozen